In [ ]:
import numpy as np
from typing import List

def image_encoder(image: List):
    features = np.array([], [])
    return features

def text_encoder(text: List):
    features = np.array([], [])
    return features

#pseudocode of CLIP method by myself
image_list = []
text_list = []
t = []
#image encoder is a way in which an image is represented by the same mathematical expression as text encoder does
image_features = image_encoder(image_list)
text_features = text_encoder(text_list)
image_features = np.linalg.norm(image_features)
text_features = np.linalg.norm(text_features)
#copute the similarity between image and text features
feature_matrix = image_features @ text_features.T
#logits here is scaled pairwise cosine similarities [n, n]
logits = feature_matrix * np.exp(t)

In [ ]:
#pesudocode of CLIP method in the thesis
#image_encoder - ResNet or Vision Transformer
#text_encoder  - CBOW or Text Transformer

Given a batch of N (image, text) pairs, the goal of CLIP is to learn a visual model that can encode images and text in a way that allows for meaningful cross-modal comparisons.

In order to achieve this, CLIP uses a contrastive learning approach, where it learns to maximize the similarity between the encoded representations of images and their corresponding text descriptions, while minimizing the similarity between different image-text pairs.

<llm-snippet-file>Learning_Transferable_Visual_Models_From_Natural_Language_Supervision.ipynb</llm-snippet-file>


# DATA FLOW OF CLIP
1. Input image data $\to$ `image_encoder` $\to$ Raw image representation $\to$ Projection $\to$ Final image embedding space
2. Input text data $\to$ `text_encoder` $\to$ Raw text representation $\to$ Projection $\to$ Final text embedding space
3. Final image embedding space @ Final text embedding space.T $\to$ Final contrastive embedding space


In [ ]:
#I[n, h, w, c] - minibatch of aligned images
#T[n, l]       - minibatch of aligned texts

#extract feature representation of each modality
I_f = image_encoder(I) #[N, d_i]
T_f = text_encoder(T)

In [ ]:
#W_i[d_i, d_e] - learned proj of image to embed
#W_t[d_t, d_e] - learned proj of text to embed

#joint multimodal embedding [n, d_e]
I_e = l2_normalize(np.dot(I_f, W_i), axis = 1)
T_e = l2_normalize(np.dot(T_f, W_t), axis = 1)

# **The thesis said: We train CLIP from scratch without initializing the image encoder with ImageNet weights or the text encoder with pre-trained weights.**

# Why did CLIP train from scratch without pre-trained weights?
**The author would like to learn powerful and generalizable visual representations directly from raw, noisy natural language supervision. If they initialized the encoder with pre-trained weights, they would not be sure that representations learned by the encoder are from models trained by themselves or pre-trained weights from Image Net or text.**

# What is the definition of contrastive embedding spaces?
**A contrastive embedding space is a giant and multidimensional "mapping room." In this room, embeddings are "Nodes" of a raw item in the real world denoted by a vector of numbers. The connections between nodes are determined by the similarity between the items they represent. The more similar two items are, the closer they are pushed by the training model. The more different the two items are, the farther they are pulled by the training model. The push and pull process over training iterations is called contrastive learning.**

# What types of projection refer to in the thesis?

There are two types of projections in CLIP:

1. **Linear projections** (e.g. $W_i$ and $W_t$)
   It is a simple matrix multiplication:
   $$ \text{FinalEmbedding} = W \cdot \text{RawRepresentation} $$

2. **Non-linear projections**
   It is a more complex bridge, typically an MLP (e.g. Linear layer $\to$ ReLU Activation $\to$ Linear layer):
   $$ \text{FinalEmbedding} = W_2 \cdot \text{ReLU}(W_1 \cdot \text{RawRepresentation}) $$


# The thesis said: For the first, we use ResNet-50 as the base architecture for the image encoder due to its **widespread adoption** and **proven performance**. We make several modifications to the original version using the ResNet-D improvements and the antialiased rec-2 blur pooling. We also replace the global average pooling layer with an attention pooling mechanism.

# What is the architecture of ResNet-50?

**ResNet-50** is a convolutional neural network architecture consisting of 50 layers. It is composed of a series of **residual blocks**, each comprising convolutional layers followed by batch normalization and ReLU activation functions. This architecture is specifically designed to address the **vanishing gradient problem** and improve the training of deep neural networks.

### The Residual Block Mechanism
A residual block allows the input of the block ($x$) to be added to the output of the block's layer ($F(x)$), creating a "**skip connection**". The output is therefore:

$$ \text{Output} = F(x) + x $$

This mechanism makes it easier for the network to learn features that are not directly present in the input.

### Advantages
1. **Enables Deep Training**: Solves the vanishing gradient problem, allowing for the creation of deeper networks.
2. **Excellent Performance**: It has achieved state-of-the-art results on many computer vision benchmarks like ImageNet.
3. **Efficient Learning**: The residual connections help the model to learn more easily during training. If a layer is not useful, the model can learn to ignore it and rely on the identity function.

### Disadvantages
1. **Computational Cost**: While powerful, a 50-layer network is computationally more expensive to train and run than shallower networks.
2. **Complexity**: The architecture is more complex than simpler sequential models like VGG.

### Theoretical Hypothesis
> We hypothesize that it is easier to optimize the residual mapping than to optimize the original, unreferenced mapping. To the extreme, if an identity mapping were optimal, it would be easier to push the residual to zero than to fit an identity mapping by a stack of nonlinear layers.

The formulation of $F(x) + x$ can be realized by feedforward neural networks with **“shortcut connections”** (He et al., 2016a).


The method of optimizing ResNet-50:
1. **ResNet-D Improvements**: Replaces the initial $7\times7$ convolution with three stacked $3\times3$ convolutions and adds a $2\times2$ average pooling layer to the skip connections (shortcut paths) in downsampling blocks to prevent information loss.
2. **Antialiased Rect-2 Blur Pooling**: Applies a low-pass filter (blurring) before downsampling operations to satisfy the sampling theorem, thereby improving shift invariance and reducing aliasing artifacts.
3. **Attention Pooling Mechanism**: Replaces the static Global Average Pooling (GAP) with a Multi-Head Attention (MHA) mechanism, enabling the model to dynamically weight spatial features (pixels) based on their semantic relevance to the global context.


# The thesis said: For the second architecture, we experiment with the recently introduced Vision Transformer (ViT)(Dosovitskiy et al.,2020). We closely follow their implementation with only the minor modification of adding an additional layer normalization to the combined patch and position embedding before the transformer and use a slightly different initialization scheme

# What is a Vision Transformer?

**The Vision Transformer (ViT)** is an architecture that applies the Transformer model, originally designed for natural language processing (NLP), to computer vision tasks. Instead of processing a sequence of words, a ViT processes a sequence of image patches.

### What is the core of the ViT?

The core of the ViT is the **self-attention mechanism**. Here's how it works:

1. **Image to Patches**: The input image is split into a grid of fixed-size square patches (e.g., $16 \times 16$ pixels).
2. **Flatten and Project**: Each patch is flattened into a long vector and then linearly projected into an "embedding," which is a vector of a specific dimension.
3. **Add Positional Information**: Because the Transformer architecture is inherently order-agnostic, "**positional embeddings**" are added to the patch embeddings. This gives the model information about the original location of each patch in the image.
4. **Self-Attention**: This sequence of patch embeddings is fed into a standard Transformer encoder. The self-attention layers allow the model to weigh the importance of every other patch when considering a single patch. For example, when the model looks at a patch containing a car's wheel, self-attention helps it pay more attention to other patches containing the car's body and windows, allowing it to understand the global context.

> **In short**, ViT's core idea is to treat an image not as a grid of pixels, but as a sequence of patches and find relationships between them using self-attention.

### What are the advantages and disadvantages of the Vision Transformer?

#### Advantages
*   **Scalability**: This is its biggest advantage. Unlike CNNs, which have a built-in "inductive bias" for local patterns, ViTs learn these spatial relationships from scratch. With massive datasets (hundreds of millions of images or more), ViTs can learn more general and powerful representations, often outperforming CNNs.
*   **Global Receptive Field**: From the very first layer, the self-attention mechanism can consider the entire image (all patches). CNNs, by contrast, build up a global view slowly through successive layers. This makes ViTs excellent at understanding long-range dependencies within an image.

#### Disadvantages
*   **Data-Hungry**: Because ViTs lack the built-in assumptions of CNNs, they require enormous amounts of data to learn basic image features. When trained on smaller datasets (like the standard ImageNet-1k), they often perform worse than well-tuned CNNs like ResNet.
*   **Less Inductive Bias**: The very thing that makes them scalable (lack of bias) is a weakness on smaller datasets. CNNs are inherently designed for images (locality, translation invariance), which gives them a head start in learning.

### What is the mathematical theorem of ViT?

Like ResNet, ViT is not based on a single formal theorem. Its core mathematical operation is the **Scaled Dot-Product Attention**, which is the heart of the self-attention mechanism. The formula is:

$$ \text{Attention}(Q, K, V) = \text{softmax}\left( \frac{Q K^T}{\sqrt{d_k}} \right) V $$

Where:
*   $Q$ (**Query**): A representation of the current patch that is "asking" for information.
*   $K$ (**Key**): Representations of all other patches that are "offering" information.
*   $V$ (**Value**): The actual content/features of the other patches.

This formula calculates a similarity score between the current patch ($Q$) and all other patches ($K$), normalizes these scores into weights, and then computes a weighted sum of all patch contents ($V$). This allows the model to create a new representation for the current patch that is informed by its global context.


# We use the Adam optimizer (Kingma & Ba, 2014) with decoupled weight decay regularization(Loshchilov & Hutter, 2017) applied to all weights that are not gains or biases, and decay the learning rate using a cosine schedule (Loshchilov & Hutter, 2016)

# How the Specific Methods Update the Weights

The methods you listed are sophisticated tools that control **step d** (the weight update) to make the "journey" faster, more efficient, and more reliable.

## 1. The Optimizer: Adam (with Decoupled Weight Decay)

**Adam** is the engine that decides how to take that step downhill. It's a very advanced version of Gradient Descent.

### What it does
Instead of just looking at the current gradient, Adam maintains two "memories":

1.  **Momentum (The Average of Past Gradients):**
    It keeps a running average of the past gradients. This is like a heavy ball rolling downhill; it builds up momentum and doesn't get stuck in small bumps or easily change direction. This helps the training process move faster and more consistently.

2.  **Adaptive Learning Rate (The Average of Squared Past Gradients):**
    It also keeps a running average of the squares of past gradients. This allows it to adapt the step size for each individual weight.
    *   If a weight's gradient is consistently large, Adam will take *smaller* steps for that weight to avoid overshooting the target.
    *   If the gradient is small and noisy, it will take *larger* steps.

### Decoupled Weight Decay (A Regularizer)

*   **Problem:** A very deep model can "**overfit**"—it can just memorize the training data instead of learning general patterns.
*   **Solution:** Weight decay is a penalty for having large weights. It encourages the model to find simpler solutions.
*   **"Decoupled":** This is a specific, improved way of applying this penalty. Instead of mixing the penalty calculation with the gradient (which can interfere with Adam's adaptive logic), it's applied as a separate, final step.

After Adam calculates the weight update, the weights are "decayed" slightly towards zero:
$$ \text{weight} = \text{weight} \times (1 - \text{decay\_rate}) $$
This makes the regularization more stable and effective.

**How it updates the weights:** For every weight in the model, Adam uses its two memories (momentum and the adaptive learning rate) to calculate a smart, individualized update. Then, the decoupled weight decay slightly shrinks the weight. This happens for all weights simultaneously.

---

## 2. The Learning Rate Scheduler: Cosine Schedule

*   **The Problem:** The "step size" (called the learning rate) is crucial. If it's too big, you'll bounce around the valley and never find the bottom. If it's too small, the training will take forever.
*   **The Solution:** A learning rate schedule changes the learning rate over time. The **cosine schedule** is a very popular and effective one.

**How it works:**
1.  It starts with a **relatively high learning rate**. This allows the model to make big progress and quickly move into the general area of the "valley" (a good solution).
2.  It then **smoothly and slowly decreases** the learning rate in the shape of a cosine curve.
3.  It ends with a **very low learning rate**. This allows the model to take tiny, careful steps to settle into the exact bottom of the valley.

---

## Summary of the Process

So, for every batch of data, the training process gets all the weights updated as follows:

1.  **Calculate Loss & Gradients:** The model makes a prediction, the loss is calculated, and the gradient (the "downhill" direction) is computed for every weight.
2.  **Adam Calculates the Update:** The Adam optimizer takes these raw gradients and, using its memory of past gradients, calculates a precise update step for each individual weight.
3.  **Learning Rate is Applied:** The size of this step is scaled by the current learning rate, which is determined by the Cosine Schedule.
4.  **Weights are Updated:** The calculated update is applied to each weight.
5.  **Weight Decay is Applied:** Finally, as a separate step, all the main weights (but not biases or special "gain" parameters) are slightly reduced by the decoupled weight decay to prevent overfitting.

This entire cycle repeats for the next batch of data, iteratively refining the millions of weights until the model becomes highly accurate.


# For each dataset, we use the names of all the classes in the dataset as the set of potential text pairings and predict the most probable (image, text) pair according to CLIP. In a bit more detail, we first compute the feature embedding of the image and the feature embedding of the set of possible texts by their respective encoders. The cosine similarity of these embeddings is then calculated,scaled by a temperature parameter τ , and normalized into a probability distribution via a softmax.

In [ ]:
#t             - learned temperature parameters
#scaled pairwise cosine similarities [n, n]
logits = np.dot(I_e, T_e.T) * np.exp(t)

In [ ]:
#symmetric loss function
labels = np.arange(n)
loss_i = cross_entropy_loss(logits, labels, axis = 0)
loss_t = cross_entropy_loss(logits, labels, axis = 1)

# The Connection to Multinomial Logistic Regression

Let's look at the standard formula for **multinomial logistic regression**, which is used for multi-class classification. The probability of an input $x$ belonging to class $i$ is given by the **softmax function**:

$$ P(y=i \mid x) = \text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}} $$

**Where:**
*   $K$ is the total number of classes.
*   $z_i$ is the "logit" or the raw score for class $i$. In classic logistic regression, this is calculated as a linear function of the input:
    $$ z_i = w_i^T x + b_i $$
    *   $x$ is the input feature vector.
    *   $w_i$ is the weight vector for class $i$.
    *   $b_i$ is the bias term for class $i$.

# How CLIP Implements This Framework

Now, let's map the **CLIP zero-shot prediction process** onto this exact mathematical structure.

1.  **The Input Feature Vector ($x$):**
    This is the L2-normalized feature embedding of the image, computed by the image encoder.
    $$ x = I_e $$

2.  **The Class Weight Vector ($w_i$):**
    This is the most brilliant part of CLIP. In a traditional model, the weight vectors $w_i$ are learned through thousands of iterations of training for a fixed set of classes. In CLIP, the L2-normalized feature embedding of the class name's text acts as the weight vector for that class.
    $$ w_i = T_{e_i} $$
    *(where $T_{e_i}$ is the text embedding for the $i$-th class name, e.g., "a photo of a dog")*.

3.  **The Logit Score ($z_i$):**
    In classic logistic regression, the logit $z_i$ is the dot product $w_i^T x$. In CLIP, it's the dot product between the image embedding and the text embedding. Because the embeddings are L2-normalized, the dot product is mathematically equivalent to the **cosine similarity**.
    $$ z_i = I_e \cdot T_{e_i}^T $$
    *(This is the cosine similarity between the image and the $i$-th text description)*.

4.  **Scaling and Normalization:**
    *   The CLIP paper adds a learned **temperature parameter $\tau$**. This scales the logits before the softmax, controlling the "sharpness" of the probability distribution. A higher temperature makes the model less confident (probabilities are closer together), while a lower temperature makes it more confident.
    *   The final probability is then calculated using the softmax function, exactly as in logistic regression:
        $$ P(\text{class} = c_i \mid \text{Image} = I) = \text{softmax}\left( (I_e \cdot T_{e_i}^T) \cdot \tau \right) $$


# The weaknesses of CLIP

1. **Need for improving computational and data efficiency of CLIP**: While scaling has so far steadily improved performance and suggests a route for continued improvement, we estimate around 1000x increase in compute is required for zero-shot CLIP to reach overall state-of-the-art performance.
2. **Poor performance on difficult tasks,such as fine-grained classification tasks, more abstract tasks and novel tasks**:When compared to task-specific models, the performance of CLIP is poor on several types of fine-grained classification such as differentiating models if cars, species of flowers and variants of aircraft. CLIP also struggles with more abstract and systematic tasks such as counting the number of objects in an image. Finally, for novel tasks which are unlikely to be included in CLIP's pre-training dataset, such as classifying the distance to the nearest car in a photo, CLIP's performance can be near random.
3. **Lower score for OCR on the handwritten digits of MNIST**: Both semantic and near-duplicate nearest-neighbor retrieval verify that there are almost no images that resemble MNIST digits in our pre-training dataset. This suggests CLIP tries to circumvent the problem and hopes that by training on such a large and varied dataset that all data will be effectively in-distribution.
4. **Limitation on composing a new text concept for unseen images**: Although CLIP can flexibly generate zero-shot classifiers for a wide variety of tasks and datasets, CLIP is still limited to choosing from only those concepts in a given zero-shot classifier.
5. **Incapability to address the poor data efficiency**: CLIP also does not address the poor data efficiency of deep learning. Instead, CLIP compensates by using a source of supervision that can be scaled to hundreds of millions of training examples.